<div class='heading'>
    <div style='float:left;'><h1>CPSC 8810: Machine Learning for Graphs</h1></div>
     <img style="float: right; padding-right: 10px" width="100" src="https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/images/clemson_paw.png"> </div>
     </div>

**Clemson University**<br>
**Instructor(s):** Aaron Masino <br>

## Lab 3: Shallow embedding methods
This notebook demonstrates the creation of node embeddings and their use in graph node classification and graph edge prediction models. It first introduces the standard data structures used in PyTorch Geometric to represent graphs. Then node2vec is used to create node embeddings for the Cora citation dataset. Finally, neural network models are created to use the embeddings to predict the article class for nodes in the Cora network and to predict if a link (citation) should exist between two nodes in the network. 

### Learning Objectives
1. Understand the elements of the data structures used to represent graphs in the PyTorch Geometric library
2. Understand the specific concerns and approaches to graph data splitting for machine learning model training and evaluation
3. Apply random walk methods to create node embeddings
4. Create machine learning models using node and graph embeddings to perform node classification and edge prediction 
5. Evaluate the performance of machine learning models that use node embeddings for prediction

In [ ]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
from pathlib import Path

# Graph analysis
import networkx as nx

# PyTorch and PyTorch Geometric
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
# import torch.nn as tnn so as not to conflict with nn from PyTorch Geometric
from torch import nn as tnn
from torch_geometric.datasets import Planetoid, KarateClub
from torch_geometric.utils import to_networkx
from torch_geometric.data import Data
from torch_geometric.nn import Node2Vec
from torch_geometric.transforms import RandomNodeSplit

# Machine Learning
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc)
from sklearn.manifold import TSNE
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Setup
plt.style.use('default')
sns.set_palette("husl")

# Create output directory
output_dir = Path('./output/lab_03')
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory created: {output_dir}")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Reusable random state
RANDOM_STATE = 654321
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Pandas set max columns to display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. PGM Graph Data

PyG uses the [torch_geometric.data.Data](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.Data.html#torch_geometric.data.Data) class to represent graph data. The `Data` class is a _dictionary like_ class in that it can hold any number of arbitrarily named objects. However, some standard objects present for most graphs are the following
- __x__ : Node feature matrix (i.e., the node attributes) `[num_nodes, node_feature_dim]`
- __edge_index__ : Edges in sparse coordinate (COO) formate `[2, num_edges]`. Note in undirecte graphs, there are two entries per edge (one for each direction.)
- __edge_attr__ : Edge attritube matrix `[num_edges, edge_feature_dim]`
- __y__ : node or graph targets (i.e., class label regression value), shape depends on task
- __pos__ : Node position matrix with shape `[num_node, num_dimensions]`, positions of nodes in a vector space (e.g., 3-D Euclidean)

For more details see also the PGM documentation on [Data Handling of Graphs](https://pytorch-geometric.readthedocs.io/en/2.5.2/get_started/introduction.html#data-handling-of-graphs)

## 1.1 Creating a Simple Graph with PyG Data

Let's create a simple example to understand the PyG Data structure. We'll build a fully connected undirected graph with 4 nodes.

**Key Points:**
- For undirected graphs, edges must be stored in both directions in `edge_index`
- Node features can be any tensor representing node attributes - although we will use these in future labs, they are not used in this lab which focuses on shallow embedding methods.
- The `Data` class includes methods to handle tensor operations and device (CPU, GPU) placement

First, we create a graph by specifying the `edge_index`. By default, the PyG `Data` class expects the edge index to be a two dimension torch tensor where the first dimension is a tensor of edge starting nodes and the second dimension is a tensor of edge ending nodes. Nodes are numbered from `0` to `n-1` in PyG.

In [ ]:
edge_index = torch.zeros((2,8), dtype=torch.long) # 2 rows for edge source and target nodes, 8 edges to account for undirected edges
edge_index[0] = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3]) # source nodes
edge_index[1] = torch.tensor([1, 2, 0, 3, 0, 3, 1, 2]) # target nodes

data = Data(edge_index=edge_index)

print("=== Dataset Information ===")
print(f"Number of graphs in dataset: {len(data)}")
print("Keys in the dataset:", data.keys())

print('Is graph directed: ',data.is_directed())  # Check if the graph is directed

The two dimensional edge tensor representation is not always convenient. Sometimes, we may prefer to build an edge list where each list element is a tuple containing the index of the start and stop nodes of the edge. We can use this representation by first transposing the `n X 2` tensor of edge pairs and applying use the `contiguous` method to ensure the data is stored in a contiguous block of memory - by default the transpose method returns a view of the tensor in the transposed shape that is not in a contiguous memory block. For computational efficiency, it will be important to have the tensor stored in a contigous memory block.

In [ ]:
# Create a fully connected undirected graph with 4 nodes using PyG Data
edge_list = []
num_nodes = 4
for i in range(num_nodes):
    for j in range(num_nodes):
        if i != j:  # No self-loops
            edge_list.append([i, j])

print('Edge list:', edge_list)

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print('Edge index shape:', edge_index.shape)
print('Edge index:', edge_index)

# Create the PyG Data object
data = Data(edge_index=edge_index)

Let's visualize this simple network. We can use PyG's `to_networkx` to convert the PyG data structure to a Networkx data structure and then plot using either the built in Netowrkx plotting or Graphviz.

In [ ]:
# Convert PyG Data to NetworkX for visualization
G_example = to_networkx(data, to_undirected=True)

# Create a figure with two subplots
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Left plot: Basic graph structure
pos = nx.spring_layout(G_example, seed=RANDOM_STATE)  # positions for all nodes
nx.draw(G_example, pos, ax=ax, with_labels=True, node_color='lightblue', 
        node_size=800, font_size=16, font_weight='bold')
ax.set_title("4-Node Fully Connected Graph")

plt.tight_layout()
plt.show()

We can also construct directed networks in PyG. Let's build a simple example as above but with directed edges from 0->1, 1->2, 2->3, 3->0, to form a cycle.

In [ ]:
edge_list = []
num_nodes = 4
for i in range(num_nodes):
    if i==num_nodes-1:
        edge_list.append([i, 0])
    else:
        edge_list.append([i, i+1])  # Directed edges forming a cycle

print('Edge list:', edge_list)

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print('Edge index shape:', edge_index.shape)

data = Data(edge_index=edge_index)
print('Is graph directed: ',data.is_directed())

<p style="font-size: 22px; color: Tomato"><strong>Excercise 1.1</strong></p>

Complete the code cell below to visualize the undirected graph created above. Careful with the `to_undirected` input as this is a directed network

In [ ]:
# Convert PyG Data to NetworkX for visualization
G_example = ___ # YOUR CODE HERE

# Create a figure with two subplots
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Left plot: Basic graph structure
pos = nx.spring_layout(G_example, seed=RANDOM_STATE)  # positions for all nodes
nx.draw(G_example, pos, ax=ax, with_labels=True, node_color='lightblue', 
        node_size=800, font_size=16, font_weight='bold')
ax.set_title("4-Node Fully Connected Graph")

plt.tight_layout()
plt.show()

## 1.2 Loading Real Graph Data with PyG

The PyG libaray includes a variety of [benchmark datasets](https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html) that can be used for model development and method evaluation. We will use several of these throughout the course. 

Let's load the Karate Club dataset to see the contents of a more realisitc dataset. Notice that:
- `len(karate_dataset)` = 1, the number of graphs in the dataset. Many datasets include multiple graphs (more on this below)
- The graph in the dataset includes a `train_mask` element in addition to the standard elements described previously. We will use masks to determine which nodes, edges, or graphs are to be used for training, validation, and testing. Many of the library datasets include standard masks, though sometimes we'll need to create these (more on this below).

In [ ]:
# Load the Karate Club dataset from PyG
karate_dataset = KarateClub()
karate_data = karate_dataset[0]  # Get the first (and only) graph

print("=== Karate Club Dataset ===")
print(f"Number of graphs in dataset: {len(karate_dataset)}")
print("Keys in the dataset:", karate_data.keys())

print("\n=== Karate Club Dataset's First (and only) Graph ===")
print(f"Number of nodes: {karate_data.num_nodes}")
print(f"Number of edges: {karate_data.num_edges}")
print(f"Number of node features: {karate_data.num_node_features}")
print(f"Number of classes: {karate_dataset.num_classes}")
print(f"Is undirected: {karate_data.is_undirected()}")

print(f"\nNode labels (first 10): {karate_data.y[:10]}")

print(f"Edge index shape: {karate_data.edge_index.shape}")
print(f"Train mask shape: {karate_data.train_mask.shape}")
print(f"Nodes in training set: {karate_data.train_mask.sum().item()}")

---
# 2. Random Walk Embeddings on Graph Nodes

Random walk embedding methods, such as **node2vec**, create low-dimensional vector representations of nodes by learning from sequences of nodes visited during random walks on the graph. These embeddings capture both local neighborhood structure and global graph topology.

**Key Concepts:**
- **Random walks**: Sequences of nodes visited by following edges randomly
- **node2vec**: Extends DeepWalk with biased random walks (return parameter p, in-out parameter q)
- **Embeddings**: Dense vector representations that preserve graph structure
- **Node classification**: Predicting node labels using learned embeddings

We'll use the [Cora citation dataset](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.Planetoid.html#torch_geometric.datasets.Planetoid). The dataset is taken from the  “[Revisiting Semi-Supervised Learning with Graph Embeddings](https://arxiv.org/abs/1603.08861)” paper. Each node corresponds to a published article, and the edge between two nodes represent citations.

## 2.1 Loading the Cora Dataset

Let's start by loading the dataset.

**Dataset Properties:**
- **Classes**: 7 different article categories - the categories are subsets of machine learning topics:
    1. Case Based
    2. Genetic Algorithms
    3. Neural Networks
    4. Probabilistic Methods
    5. Reinforcement Learning
    6. Rule Learning
    7. Theory
- **Features**: Each node has a feature vector that is the bag-of-words embedding for the article using a dictionary of 1,433 words. We will not use these features in this lab, but will revisit them in a future lab.

In [ ]:
cora_dataset = Planetoid(root='./data/Cora', name='Cora')
cora_data = cora_dataset[0]  # Get the first (and only) graph
cora_labels = ['Case Based', 'Genetic Algorithms', 'Neural Networks',
               'Probabilistic Methods', 'Reinforcement Learning',
               'Rule Learning', 'Theory']

print("=== Cora Dataset ===")
print(f"Number of graphs in dataset: {len(cora_dataset)}")
print("Keys in the dataset:", cora_data.keys())

print("\n=== Cora Dataset's First (and only) Graph ===")
print(f"Number of nodes: {cora_data.num_nodes}")
print(f"Number of edges: {cora_data.num_edges}")
print(f"Number of node features: {cora_data.num_node_features}")
print(f"Number of classes: {cora_dataset.num_classes}")
print(f"Is undirected: {cora_data.is_undirected()}")

print(f"\nNode labels (first 10): {cora_data.y[:10]}")

print(f"Edge index shape: {cora_data.edge_index.shape}")
print(f"Train mask shape: {cora_data.train_mask.shape}")
print(f"Nodes in training set: {cora_data.train_mask.sum().item()}")

Now let's examine the class distribution. It will be useful to know if there is class imbalance for when we train the node label prediction models.

In [ ]:
# Get label counts and percentages
label_counts = torch.bincount(cora_data.y)
percentages = (label_counts / cora_data.num_nodes * 100).numpy()

# Create bar plot
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(label_counts)), label_counts.numpy(), 
               alpha=0.7, edgecolor='black', color=[f'C{i}' for i in range(len(label_counts))])

# Add value labels on bars
for i, (bar, count, pct) in enumerate(zip(bars, label_counts, percentages)):
    height = bar.get_height()-100
    plt.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

plt.title('Cora Dataset: Node Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Cora Class', fontsize=12)
plt.ylabel('Number of Nodes', fontsize=12)
plt.xticks(range(len(label_counts)), cora_labels, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add total count
plt.text(0.02, 0.98, f'Total Nodes: {cora_data.num_nodes}', 
         transform=plt.gca().transAxes, fontsize=12, 
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

Now, let's examine the node degree statistics and the network density. These will help us set some of the parameters in the node2vec algorithm, specifically the walk length and context window. We will base these parameters on the following heuristics:

**Walk Length**

Typical walk length range: 10-100 steps.

The **walk length** should be set based on:
- A good starting point is 2-3 times the network's average shortest path length. For large networks, this is computationally expensive to calculate. We can take a random sample of nodes from the network to get an estimate.
- Application needs: If you need to capture long-range dependencies or global network properties, use longer walks (50-100). For local neighborhood structure, shorter walks (10-30) often suffice.

**Context Window Size**
- Typical range: 5-20, with 10 being a common default.

The **context window** should consider:
- Walk length relationship: Generally set to walk_length/4 to walk_length/2. A window that's too large relative to walk length can introduce noise.
- Average degree: High-degree nodes benefit from larger windows to capture their diverse neighborhoods.

In [ ]:
# Explore basic graph properties: degree statistics and diameter
print("=== Basic Graph Properties ===")

# Convert to NetworkX for analysis (treat as undirected for basic properties)
G_cora = to_networkx(cora_data, to_undirected=False)

# Degree statistics
degrees = [d for n, d in G_cora.degree()]
degree_array = np.array(degrees)
print(f"  Average degree: {np.mean(degree_array):.2f}")
print(f"  Degree standard deviation: {np.std(degree_array):.2f}")
print(f"  Median degree: {np.median(degree_array):.2f}")

# Sample shortest paths and compute average shortest path length to estimate diameter
sample_nodes = np.random.choice(cora_data.num_nodes, size=200, replace=False)
shortest_paths = []
for i in range(len(sample_nodes)):
    for j in range(i + 1, len(sample_nodes)):
        if sample_nodes[i] != sample_nodes[j]:
            try:
                path_length = nx.shortest_path_length(G_cora, source=sample_nodes[i], target=sample_nodes[j])
                shortest_paths.append(path_length)
            except nx.NetworkXNoPath:
                pass

avs = np.mean(shortest_paths)
print(f"\nAverage shortest path length (sampled) from {len(sample_nodes)} samples: {avs:.2f}")

# Based on heuristics:
print(f"\nSuggested walk lengths: [{int(2*avs)}, {int(3*avs)}]")
print(f"\nSuggested context window:")
print(f"\tWalk length {int(2*avs)}: [{int(avs/2)}, {int(avs)}]")
print(f"\tWalk length {int(3*avs)}: [{int(3*avs/4)}, {int(3*avs/2)}]")

## 2.2 Generating Node Embeddings with Node2Vec

Node2Vec creates node embeddings by performing biased random walks and learning representations that predict the context of nodes in these walks. The key parameters are:

- **embedding_dim**: Dimensionality of the embedding vectors
- **walk_length**: Length of each random walk 
- **context_size**: Size of the context window for skip-gram
- **walks_per_node**: Number of random walks starting from each node
- **p**: Return parameter (controls likelihood of revisiting a node)
- **q**: In-out parameter (controls exploration vs exploitation)

Here, we provide two options for these settings. The first, `lab_parameters` are the ones we'll use today. These were selected to allow for a short training time for demonstration purposes, though they may not give high quality embeddings. The second set, `exploration_parameters` are intended for you to try at home. Node2vec training will take longer with these settings but should provide higher quality embeddings. You are of course encouraged to try other settings.

In the code cell below, toggle the `USE_LAB_PARAMETERS` boolean to switch between the `lab_parameters` option and the `exploration_parameters` option.

In [ ]:
# Initialize Node2Vec model with lab-optimized parameters
print("=== Node2Vec Configuration ===")

# Lab parameters (fast execution for demonstration)
lab_params = {
    'embedding_dim': 32,
    'walk_length': 20, 
    'context_size': 5,
    'walks_per_node': 20,
    'p': 1.0,
    'q': 1.0,
    'num_negative_samples': 10
}

# Better parameters for further exploration (commented out for lab speed)
exploration_params = {
    'embedding_dim': 64,
    'walk_length': 20,
    'context_size': 5, 
    'walks_per_node': 50,
    'p': 1.0,
    'q': 1.0,
    'num_negative_samples': 10
}

# Create Node2Vec model with lab parameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nUsing device: {device}")

USE_LAB_PARAMS = True  # Set to False to use exploration parameters
params = lab_params if USE_LAB_PARAMS else exploration_params

# Clear CUDA cache to avoid memory issues
torch.cuda.empty_cache()

# Initialize Node2Vec model
node2vec_model = Node2Vec(
    cora_data.edge_index,
    **params,
    sparse=True
).to(device)

print(f"Node2Vec model created with {node2vec_model.embedding_dim} dimensional embeddings")
print(f"Model parameters: {sum(p.numel() for p in node2vec_model.parameters()):,}")

Now let's train our node2vec model. We will create to utility functions, `train_node2vec` and `plot_embedding_distribution`. As with the `node2vec` model settings, we define two sets of training parameters. The `USE_LAB_PARAMS` variable is used to control the number of training epochs (loops over the training data batches) and the batch size. We examine the distribution of embedding values to ensure they are well behaved.

In [ ]:
# Train the Node2Vec model
print("=== Training Node2Vec Model ===")

def train_node2vec(model, loader, optimizer, num_epochs):
    node2vec_model.train()
    losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0
        num_batches = 0
    
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = node2vec_model.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
        
            epoch_loss += loss.item()
            num_batches += 1
    
        avg_loss = epoch_loss / num_batches
        losses.append(avg_loss)
    
        if (epoch + 1) % 2 == 0:  # Print every 2 epochs
            print(f"  Epoch {epoch + 1:2d}/{num_epochs}: Loss = {avg_loss:.4f}")
    return losses

def plot_embedding_distribution(embeddings, node_id=0):
    # plt.subplot(1, 1, 21)
    plt.hist(embeddings[node_id].cpu().detach().numpy(), bins=20, alpha=0.7, edgecolor='black')
    plt.title(f'Distribution of Node {node_id} Embedding Values')
    plt.xlabel('Embedding Value')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Training parameters (reduced for lab demonstration)
if USE_LAB_PARAMS:
    num_epochs = 20  # For lab demonstration, use fewer epochs
    batch_size = 256  # For lab demonstration, use larger batch size
else:
    # Exploration parameters    
    num_epochs = 20  # For exploration: use 100+ epochs
    batch_size = 64  # For exploration: use smaller batch size

learning_rate = 0.01

print(f"Training parameters:")
print(f"  Epochs: {num_epochs} (for exploration: use 100+ epochs)")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")

# Setup optimizer
optimizer = torch.optim.SparseAdam(list(node2vec_model.parameters()), lr=learning_rate)

# Create data loader
loader = node2vec_model.loader(batch_size=batch_size, shuffle=True, num_workers=0)
losses = train_node2vec(node2vec_model, loader, optimizer, num_epochs)

print(f"Training completed!")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses, 'b-', linewidth=2)
plt.title('Node2Vec Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)

# Generate embeddings for all nodes
print(f"\nGenerating embeddings for all {cora_data.num_nodes} nodes...")
node2vec_model.eval()
with torch.no_grad():
    embeddings = node2vec_model()

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding sample for node 0: {embeddings[0][:5]}...")  # Show first 5 dimensions

plt.subplot(1, 2, 2)
plot_embedding_distribution(embeddings, node_id=0)

plt.tight_layout()
plt.show()

## 2.3 Predicting Node Attributes with Embeddings
In this section, we will build a neural network model that infers the node class label using the node2vec embedding as input. 

### 2.3.1 Visualizing the Node Embedding Space with TSNE
In general, we would like to get a sense of the quality of the embeddings relative to a downstream task before we build a model for that task. Let's say we are interested in using the embeddings to predict class labels on nodes in the network. Why? Imagine that in a very large graph, human labelers manually labeled a fraction of the network. We would like to be able to label the rest of it. Before putting effot into developing the model, we'd like some indication the embeddings will be useful for the task.

One way to examine the node embedding quality is plot a t-Distributed Stochastic Neighbor Embedding (t-SNE) of the embeddings (yes, an embedding of embeddings) using the scikit-learn [TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) implementation. If the node2vec embeddings are to be useful for classification, we should expect nodes in the same class to cluster together in the TSNE plot.

In [ ]:
# t-SNE visualization of node embeddings with stratified sampling
print("=== t-SNE Visualization of Node Embeddings ===")

# Take a subset of embeddings for visualization
# Create stratified sample of 500 embeddings to maintain class distribution
sample_size = 500
labels = cora_data.y.cpu().numpy()

# Calculate number of samples per class to maintain distribution
label_counts = np.bincount(labels)
total_nodes = len(labels)

samples_per_class = {}
sample_indices = []

print(f"Creating stratified sample of {sample_size} nodes:")
for class_id in range(len(label_counts)):
    class_proportion = label_counts[class_id] / total_nodes
    class_samples = max(1, int(sample_size * class_proportion))  # At least 1 sample per class
    samples_per_class[class_id] = class_samples
    
    # Get indices for this class
    class_indices = np.where(labels == class_id)[0]
    
    # Randomly sample from this class
    np.random.seed(RANDOM_STATE)
    if len(class_indices) >= class_samples:
        selected_indices = np.random.choice(class_indices, class_samples, replace=False)
    else:
        selected_indices = class_indices  # Use all if fewer than needed
    
    sample_indices.extend(selected_indices)
    print(f"  Class {class_id}: {class_samples} samples ({class_samples/sample_size*100:.1f}%)")

# Convert to numpy array and ensure we have exactly sample_size samples
sample_indices = np.array(sample_indices)
if len(sample_indices) > sample_size:
    sample_indices = sample_indices[:sample_size]

print(f"Total sample size: {len(sample_indices)}")

# Extract embeddings and labels for selected samples
sample_embeddings = embeddings[sample_indices].cpu().detach().numpy()
sample_labels = labels[sample_indices]

# Apply t-SNE
print(f"\nApplying t-SNE to {sample_embeddings.shape[0]} embeddings...")
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    max_iter=1000,
    random_state=RANDOM_STATE,
    init='pca'
)

tsne_embeddings = tsne.fit_transform(sample_embeddings)

# Create visualization
plt.figure(figsize=(12, 8))

# Plot each class with different colors
colors = [f'C{i}' for i in range(len(label_counts))]
for class_id in range(len(label_counts)):
    mask = sample_labels == class_id
    if np.sum(mask) > 0:
        plt.scatter(tsne_embeddings[mask, 0], tsne_embeddings[mask, 1], 
                   c=colors[class_id], label=f'Class {class_id}', 
                   alpha=0.7, s=60)

plt.title('t-SNE Visualization of Node2Vec Embeddings\n(Stratified Sample of 100 Nodes)', 
          fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

# Add text box with sample information
sample_info = f'Sample size: {len(sample_indices)}\n'
for class_id in range(len(label_counts)):
    class_count = np.sum(sample_labels == class_id)
    sample_info += f'Class {class_id}: {class_count} nodes\n'

plt.text(0.02, 0.98, sample_info.strip(), transform=plt.gca().transAxes, 
         fontsize=10, verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

We see above that we get reasonable clustering and hence expect the node2vec node embeddings to be useful for predicting the class membership for the Cora dataset. Let's build a model to find out.

### 2.3.2 Data Splitting with RandomNodeSplit

For machine learning evaluation, we need to split our data into training, validation, and test sets. The Cora dataset includes training, validation, and test masks. However, for historical reasons, the masks result in a non-standard split of the dataset. We will instead prefer to use a more standard data split of approximately 70% training, 10% validation, and 20% test. PyG provides the `RandomNodeSplit` class to create these splits for node-level tasks.

**Key Points:**
- **Default Cora splits**: The dataset comes with predefined train/val/test masks that result in non-standard ML splits
- **Custom splits**: We want a typical 70/10/20 split.
- **RandomNodeSplit**: Randomly assigns nodes to train/val/test sets while maintaining class balance

In [ ]:
# First, let's examine the default Cora data splits

print("=== Default Cora Data Splits ===")
print(f"Training nodes: {cora_data.train_mask.sum().item()}")
print(f"Validation nodes: {cora_data.val_mask.sum().item()}")
print(f"Test nodes: {cora_data.test_mask.sum().item()}")
print(f"Total nodes: {cora_data.num_nodes}")

# Configuration variable to switch between default and lab splits
USE_LAB_SPLITS = True  # Set to False to use default Cora splits

if USE_LAB_SPLITS:
    print(f"\n=== Creating Lab Data Splits  ===")
    
    # Lab splits: smaller training set for faster demonstration
    lab_split_params = {
        'num_train_per_class': 10,  # 10 samples per class for training, if there are extra samples, they will be added to the training set
        'num_val': 250,             # 250 samples for validation
        'num_test': 600            # 600 samples for test
    }
    
    print(f"Lab split parameters:")
    for key, value in lab_split_params.items():
        print(f"  {key}: {value}")
    
    # Create the random node split transform
    transform = RandomNodeSplit(**lab_split_params, split='train_rest')
    
    # Apply the transform to create new masks
    cora_data_split = transform(cora_data.clone())
    
    # Extract the new masks
    train_mask = cora_data_split.train_mask
    val_mask = cora_data_split.val_mask
    test_mask = cora_data_split.test_mask
    
    print(f"\nLab split results:")
    print(f"Training nodes: {train_mask.sum().item()}")
    print(f"Validation nodes: {val_mask.sum().item()}")
    print(f"Test nodes: {test_mask.sum().item()}")
    
    # Show class distribution in lab training set
    lab_train_labels = cora_data.y[train_mask]
    lab_train_label_counts = torch.bincount(lab_train_labels)
    print(f"\nLab training set class distribution:")
    for i, count in enumerate(lab_train_label_counts):
        percentage = count / lab_train_labels.shape[0] * 100
        print(f"  Class {i} ({cora_labels[i]}): {count} ({percentage:.1f}%)")
        
else:
    print(f"\n=== Using Default Cora Data Splits ===")
    # Use the default masks
    train_mask = cora_data.train_mask
    val_mask = cora_data.val_mask  
    test_mask = cora_data.test_mask

### 2.3.3 Cora Node Classification Network
Here, we will construct a neural network to infer the class membership for an input node from the Cora network. The input to the model will be the embedding produced by node2vec. Our first model will have a single layer. Our second model will use two layers. But first, let's create some utility functions to support model training and evaluation. Here, we us on PyTorch and PyG classes to train the model which means we need to track metrics manually, manage the training batches, and ensure tensors are on the correct device. (In our next lab, we'll introduce PyTorch Lightning which will manage many of these concerns for us automatically.)

In [ ]:
# Reusable functions for model training and evaluation

def train_classifier(model, X_train, y_train, X_val, y_val, num_epochs, learning_rate=0.01, batch_size=64, verbose=True):
    """
    Train a PyTorch classifier with given data using mini-batches
    
    Args:
        model: PyTorch model to train
        X_train, y_train: Training data and labels
        X_val, y_val: Validation data and labels  
        num_epochs: Number of training epochs
        learning_rate: Learning rate for optimizer
        batch_size: Size of mini-batches for training
        verbose: Whether to print training progress
    
    Returns:
        train_losses, val_losses, val_accuracies: Training history
    """
    
    # Get device from model
    device = next(model.parameters()).device
    
    # Ensure all data is on the correct device
    X_train = X_train.to(device)
    y_train = y_train.to(device)
    X_val = X_val.to(device)
    y_val = y_val.to(device)
    
    criterion = tnn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Create data loaders for batched training
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0
        num_batches = 0
        
        for batch_X, batch_y in train_loader:
            # Ensure batch data is on correct device (should already be, but just in case)
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            # zero the gradients before backward pass
            optimizer.zero_grad()
            train_out = model(batch_X)
            train_loss = criterion(train_out, batch_y)
            train_loss.backward()
            # Update model parameters by stepping in the negative gradient direction
            optimizer.step()
            
            epoch_train_loss += train_loss.item()
            num_batches += 1
        
        avg_train_loss = epoch_train_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation phase (only at end of epoch)
        model.eval()
        # compute validation loss and accuracy, we don't need gradients for validation
        with torch.no_grad():
            val_out = model(X_val)
            val_loss = criterion(val_out, y_val)
            val_pred = val_out.argmax(dim=1)
            val_acc = (val_pred == y_val).float().mean()
        
        val_losses.append(val_loss.item())  
        val_accuracies.append(val_acc.item())
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d}: Train Loss = {avg_train_loss:.4f}, Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")
    
    return train_losses, val_losses, val_accuracies

def plot_roc_curves(y_true, y_pred_proba, class_names, title="ROC Curves"):
    """
    Plot ROC curves for multi-class classification
    
    Args:
        y_true: True labels (numpy array or tensor)
        y_pred_proba: Predicted probabilities for each class (numpy array or tensor)
        class_names: List of class names
        title: Plot title
    """
    
    # Convert to numpy if tensors
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().detach().numpy()
    if torch.is_tensor(y_pred_proba):
        y_pred_proba = y_pred_proba.cpu().detach().numpy()

    # Binarize the labels for multi-class ROC
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    
    plt.figure(figsize=(10, 8))
    colors = [f'C{i}' for i in range(len(class_names))]
    
    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i], lw=2, 
                 label=f'{class_names[i]} (AUC = {roc_auc:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.500)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    """
    Plot confusion matrix with class names
    
    Args:
        y_true: True labels (numpy array or tensor)
        y_pred: Predicted labels (numpy array or tensor)
        class_names: List of class names
        title: Plot title
    """
    
    # Convert to numpy if tensors
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()
    
    cm = confusion_matrix(y_true, y_pred, normalize='pred')
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

Now we're ready to define our model and train it. As with the node2vec model, we provide two sets of model and training parameters to support in class presentation (optimized for speed) and further exploration (optimized for performance). Toggle the `USE_LAB_TRAINING` boolean to switch between the two settings. You should expect better performance with the exploration parameters, especially if you have also used the exploration parameters to create higher quality node2vec embeddings.

Our initial model is single linear layer with rectified linear unit (ReLU) activation function. The output layer has seven units - one per class lable. The cross entropy loss function (see the train classifier method above) automatically handles softmax conversion so we do not need to perform that step here.

In [ ]:
# Define single-layer neural network for node classification
class SingleLayerClassifier(tnn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = tnn.Linear(input_dim, hidden_dim)
        self.relu = tnn.ReLU()
        self.fc2 = tnn.Linear(hidden_dim, num_classes)
        self.dropout = tnn.Dropout(0.2)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Training parameters
USE_LAB_TRAINING = True  # Set to False for exploration parameters

if USE_LAB_TRAINING:
    # Lab parameters (fast training)
    hidden_dim = 32 # set to the same as the Node2Vec embedding dimension for lab
    num_epochs = 5
    learning_rate = 0.01
    print("Using lab training parameters (fast):")
else:
    # Exploration parameters (better performance)
    hidden_dim = 64 # set to the same as the Node2Vec embedding dimension for exploration
    num_epochs = 10
    learning_rate = 0.001
    print("Using exploration training parameters (better performance):")

print(f"  Hidden dimensions: {hidden_dim}")
print(f"  Training epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Device: {device}")

# Create the model
input_dim = embeddings.shape[1]  # Node2Vec embedding dimension
num_classes = len(cora_labels)   # Number of Cora classes

model1 = SingleLayerClassifier(input_dim, hidden_dim, num_classes).to(device)
print(f"\nSingle-layer model created:")
print(f"  Input dimension: {input_dim}")
print(f"  Hidden dimension: {hidden_dim}")
print(f"  Output classes: {num_classes}")
print(f"  Total parameters: {sum(p.numel() for p in model1.parameters()):,}")
print(f"  Model device: {next(model1.parameters()).device}")

# Prepare training data - ensure everything is on the correct device
X_train = embeddings[train_mask].detach().to(device)
y_train = cora_data.y[train_mask].to(device)
X_val = embeddings[val_mask].detach().to(device)
y_val = cora_data.y[val_mask].to(device)
X_test = embeddings[test_mask].detach().to(device)
y_test = cora_data.y[test_mask].to(device)

Now let's train the model and evaluate its performance during training and validation.

In [ ]:
# Train the single-layer model
print("=== Training Single-Layer Model ===")

# Set batch size based on training configuration
if USE_LAB_TRAINING:
    batch_size = 32  # Smaller batch for lab demonstration
else:
    batch_size = 64  # Standard batch size for exploration

print(f"Using batch size: {batch_size}")

train_losses1, val_losses1, val_accuracies1 = train_classifier(
    model1, X_train, y_train, X_val, y_val, num_epochs, learning_rate, batch_size
)

print(f"\nTraining completed!")

# Plot training curves
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses1, label='Training Loss', color='blue')
plt.plot(val_losses1, label='Validation Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(val_accuracies1, label='Validation Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Now, assuming we're satisfied with our model setup, let's evaluate it on the test set. If using the `USE_LAB_PARAMATERS=True` for node2vec training and `USE_LAB_TRAINING=True` for network model training, we should get reasonablly good performance. However, you should get better performance setting both values to `False` and retraining with the exploration parameters.

In [ ]:
# Detailed evaluation of single-layer model
print("=== Single-Layer Model Evaluation ===")

# test predictions and probabilities
model1.eval()
with torch.no_grad():
    test_out1 = model1(X_test)
    test_proba1 = torch.softmax(test_out1, dim=1)
    test_pred1 = test_out1.argmax(dim=1)

# Convert tensors to numpy arrays, ensuring they're on CPU first
y_test_np = y_test.cpu().numpy()
test_pred1_np = test_pred1.cpu().numpy()
test_proba1_np = test_proba1.cpu().numpy()

print("Classification Report:")
print(classification_report(y_test_np, test_pred1_np, target_names=cora_labels, digits=4))

# ROC curves for each class - functions now handle device conversion automatically
plot_roc_curves(y_test, test_proba1, cora_labels, 
                title="ROC Curves - Single-Layer Model")

# Confusion matrix - functions now handle device conversion automatically
plot_confusion_matrix(y_test, test_pred1, cora_labels,
                     title="Confusion Matrix - Single-Layer Model")

<p style="font-size: 22px; color: Tomato"><strong>Excercise 2.1</strong></p>
Now it's your turn! Complete the implementation of a two-layer neural network where the second layer is half the size of the first layer.

**Requirements:**
- First hidden layer: Same size as single-layer model (`hidden_dim`)
- Second hidden layer: Half the size of the first (`hidden_dim // 2`)
- Use ReLU activation after each hidden layer
- Include dropout for regularization

In [ ]:
# Exercise 2.1: Complete the two-layer neural network implementation
class TwoLayerClassifier(tnn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, num_classes):
        super().__init__()
        # TODO: Define the layers
        # YOUR CODE HERE
        self.fc1 = ___  # First linear layer: input_dim -> hidden_dim1
        self.relu1 = ___  # First ReLU activation
        self.dropout1 = ___  # First dropout layer
        
        self.fc2 = ___  # Second linear layer: hidden_dim1 -> hidden_dim2
        self.relu2 = ___  # Second ReLU activation  
        self.dropout2 = ___  # Second dropout layer
        
        self.fc3 = ___  # Output layer: hidden_dim2 -> num_classes
        
    def forward(self, x):
        # TODO: Implement the forward pass
        # YOUR CODE HERE
        x = ___  # Apply first linear layer
        x = ___  # Apply first ReLU
        x = ___  # Apply first dropout
        
        x = ___  # Apply second linear layer
        x = ___  # Apply second ReLU
        x = ___  # Apply second dropout
        
        x = ___  # Apply output layer
        return x

# Create the two-layer model
hidden_dim1 = hidden_dim      # Same as single-layer model
hidden_dim2 = hidden_dim // 2 # Half the size of first layer

model2 = TwoLayerClassifier(input_dim, hidden_dim1, hidden_dim2, num_classes).to(device)
print(f"Two-layer model created:")
print(f"  Input dimension: {input_dim}")
print(f"  Hidden dimension 1: {hidden_dim1}")
print(f"  Hidden dimension 2: {hidden_dim2}")
print(f"  Output classes: {num_classes}")
print(f"  Total parameters: {sum(p.numel() for p in model2.parameters()):,}")

Now, let's train the two layer model. Notice that we can reuse the `train_classifier` method to train the model.

In [ ]:
# Train the two-layer model
print("=== Training Two-Layer Model ===")

train_losses2, val_losses2, val_accuracies2 = train_classifier(
    model2, X_train, y_train, X_val, y_val, num_epochs, learning_rate, batch_size
)

print(f"\nTraining completed!")

# Evaluate on test set
model2.eval()
with torch.no_grad():
    test_out2 = model2(X_test)
    test_pred2 = test_out2.argmax(dim=1)
    test_acc2 = (test_pred2 == y_test).float().mean()
    test_proba2 = torch.softmax(test_out2, dim=1)

print(f"Test accuracy: {test_acc2:.4f}")

# Compare model performance
test_acc1 = (test_pred1 == y_test).float().mean()
print(f"\n=== Model Comparison ===")
print(f"Single-layer model test accuracy: {test_acc1:.4f}")
print(f"Two-layer model test accuracy: {test_acc2:.4f}")
improvement = test_acc2 - test_acc1
print(f"Improvement: {improvement:.4f} ({improvement/test_acc1*100:+.2f}%)")

# Plot comparison of training curves
plt.figure(figsize=(15, 10))

# Loss comparison
plt.subplot(2, 3, 1)
plt.plot(train_losses1, label='Single-layer Train', color='blue', linestyle='-')
plt.plot(val_losses1, label='Single-layer Val', color='blue', linestyle='--')
plt.plot(train_losses2, label='Two-layer Train', color='red', linestyle='-')
plt.plot(val_losses2, label='Two-layer Val', color='red', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Accuracy comparison
plt.subplot(2, 3, 2)
plt.plot(val_accuracies1, label='Single-layer', color='blue')
plt.plot(val_accuracies2, label='Two-layer', color='red')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Test accuracy comparison
plt.subplot(2, 3, 3)
plt.bar(['Single-Layer', 'Two-Layer'], 
        [test_acc1.cpu().numpy(), test_acc2.cpu().numpy()], 
        color=['lightblue', 'lightcoral'], edgecolor='black')
plt.ylabel('Test Accuracy')
plt.title('Test Performance Comparison')
plt.ylim([0, 1])
plt.grid(True, alpha=0.3, axis='y')

# Individual model training curves
plt.subplot(2, 3, 4)
plt.plot(train_losses1, label='Training Loss', color='blue')
plt.plot(val_losses1, label='Validation Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Single-Layer Model')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 5)
plt.plot(train_losses2, label='Training Loss', color='blue')
plt.plot(val_losses2, label='Validation Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Two-Layer Model')
plt.legend()
plt.grid(True, alpha=0.3)

# Parameter comparison
plt.subplot(2, 3, 6)
params1 = sum(p.numel() for p in model1.parameters())
params2 = sum(p.numel() for p in model2.parameters())
plt.bar(['Single-Layer', 'Two-Layer'], [params1, params2], 
        color=['lightblue', 'lightcoral'], edgecolor='black')
plt.ylabel('Number of Parameters')
plt.title('Model Complexity Comparison')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

What do you notice about the training and validation performance of the two layer model compared to the one layer model?

Finally, let's examine the test set performance. If using the `USE_LAB_PARAMATERS=True` for node2vec training and `USE_LAB_TRAINING=True` for network model training, the two layer model will do about the same or maybe worse than the single layer model. Essentially, the embedding quality is insufficient for this more complex model to learn with the current dataset. However, if you rerun the node2vec and two layer model training using the exploration parameters (setting both variables to `False`), the two layer model performance should be slightly better than the single layer. Do you think the perforamnce increase is worth the additional model complexity?

In [ ]:
# Classification report for two-layer model
print("=== Two-Layer Model Evaluation ===")

# Convert tensors to numpy, ensuring they're on CPU first
y_test_np = y_test.cpu().numpy()
test_pred2_np = test_pred2.cpu().numpy()
test_proba2_np = test_proba2.cpu().numpy()

print("Classification Report:")
print(classification_report(y_test_np, test_pred2_np, target_names=cora_labels, digits=4))

<p style="font-size: 22px; color: Tomato"><strong>Excercise 2.2</strong></p>
Evaluation Plots for Two-Layer Model

Complete the code below to generate ROC curves and confusion matrix for the two-layer model. Use the reusable functions we defined earlier.

In [ ]:
# Exercise 2.2: Complete the evaluation plots for the two-layer model

# TODO: Generate ROC curves for the two-layer model
# Use the plot_roc_curves function with the correct parameters
# YOUR CODE HERE
plot_roc_curves(___, ___, ___, title="___")

# TODO: Generate confusion matrix for the two-layer model  
# Use the plot_confusion_matrix function with the correct parameters
# YOUR CODE HERE
plot_confusion_matrix(___, ___, ___, title="___")

---
## 2.4 Predicting Links with Embeddings

Link prediction is the task of predicting whether an edge should exist between two nodes in a graph. This is useful for:
- **Recommender systems**: Suggesting connections between users
- **Drug discovery**: Predicting interactions between compounds
- **Knowledge graphs**: Inferring missing relationships
- **Social networks**: Recommending friendships or collaborations

**Key Concepts:**
- **Link prediction as binary classification**: For each node pair, predict if an edge exists
- **Node pair embeddings**: Combine embeddings of two nodes to represent their relationship
- **Negative sampling**: Generate non-existent edges as negative examples
- **Cosine similarity**: Measure similarity between node embeddings

We'll use the same Cora dataset and Node2Vec embeddings from the previous section.

### 2.4.1 Analyzing Embedding Similarity for Connected vs. Random Node Pairs

Before building a link prediction model, let's analyze whether our Node2Vec embeddings capture meaningful relationships. We'll compare the cosine similarity distribution between:
1. **Connected node pairs**: Nodes that are actually connected by edges
2. **Random node pairs**: Randomly selected pairs of nodes

If the embeddings are good for link prediction, connected nodes should have higher similarity than random pairs.

In [ ]:
# Analyze cosine similarity for connected vs. random node pairs
print("=== Embedding Similarity Analysis ===")

# Get edge list (convert from PyG format)
edge_index = cora_data.edge_index
num_edges = edge_index.shape[1]

# Since Cora is undirected, we have duplicate edges. Let's get unique edges.
# Convert to undirected by only keeping edges where source < target
unique_edges = []
for i in range(num_edges):
    src, tgt = edge_index[0, i].item(), edge_index[1, i].item()
    if src < tgt:  # Only keep one direction for undirected edges
        unique_edges.append((src, tgt))

unique_edges = torch.tensor(unique_edges).t()  # Shape: [2, num_unique_edges]
print(f"Total directed edges: {num_edges}")
print(f"Unique undirected edges: {unique_edges.shape[1]}")

# Sample for analysis (use subset for computational efficiency)
num_samples = 1000
sample_indices = np.random.choice(unique_edges.shape[1], min(num_samples, unique_edges.shape[1]), replace=False)
sampled_edges = unique_edges[:, sample_indices]

print(f"Analyzing {sampled_edges.shape[1]} connected pairs...")

# Calculate cosine similarities for connected pairs
connected_similarities = []
for i in range(sampled_edges.shape[1]):
    node1, node2 = sampled_edges[0, i], sampled_edges[1, i]
    emb1 = embeddings[node1].unsqueeze(0)
    emb2 = embeddings[node2].unsqueeze(0)
    sim = F.cosine_similarity(emb1, emb2).item()
    connected_similarities.append(sim)

connected_similarities = np.array(connected_similarities)

# Generate random node pairs (same number as connected pairs)
print(f"Generating {len(connected_similarities)} random pairs...")
random_similarities = []
num_nodes = cora_data.num_nodes

for _ in range(len(connected_similarities)):
    # Sample two different random nodes
    node1, node2 = np.random.choice(num_nodes, 2, replace=False)
    emb1 = embeddings[node1].unsqueeze(0)
    emb2 = embeddings[node2].unsqueeze(0)
    sim = F.cosine_similarity(emb1, emb2).item()
    random_similarities.append(sim)

random_similarities = np.array(random_similarities)

# Calculate statistics
print(f"\n=== Similarity Statistics ===")
print(f"Connected pairs:")
print(f"  Mean: {connected_similarities.mean():.4f}")
print(f"  Std:  {connected_similarities.std():.4f}")
print(f"  Min:  {connected_similarities.min():.4f}")
print(f"  Max:  {connected_similarities.max():.4f}")

print(f"\nRandom pairs:")
print(f"  Mean: {random_similarities.mean():.4f}")
print(f"  Std:  {random_similarities.std():.4f}")
print(f"  Min:  {random_similarities.min():.4f}")
print(f"  Max:  {random_similarities.max():.4f}")

diff_mean = connected_similarities.mean() - random_similarities.mean()
print(f"\nDifference in means: {diff_mean:.4f}")

# Plot distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(connected_similarities, bins=30, alpha=0.7, label='Connected pairs', color='blue', density=True)
plt.hist(random_similarities, bins=30, alpha=0.7, label='Random pairs', color='red', density=True)
plt.xlabel('Cosine Similarity')
plt.ylabel('Density')
plt.title('Cosine Similarity Distributions')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([connected_similarities, random_similarities], 
            labels=['Connected', 'Random'])
plt.ylabel('Cosine Similarity')
plt.title('Similarity Comparison (Box Plot)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical significance test
from scipy import stats
statistic, p_value = stats.ttest_ind(connected_similarities, random_similarities)
print(f"\nStatistical test (t-test):")
print(f"  t-statistic: {statistic:.4f}")
print(f"  p-value: {p_value:.2e}")
print(f"  Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

### 2.4.2 Preparing Link Prediction Data

For link prediction, we need to create a dataset where:
1. **Positive examples**: Existing edges (label = 1)
2. **Negative examples**: Non-existing edges (label = 0)
3. **Features**: Concatenated embeddings of node pairs

We'll split the existing edges into train/val/test sets and generate negative samples for each split.

In [ ]:
# Prepare link prediction dataset
print("=== Preparing Link Prediction Data ===")

def create_negative_edges(num_nodes, positive_edges, num_negative):
    """Create negative edge samples that don't exist in positive_edges"""
    positive_set = set()
    for i in range(positive_edges.shape[1]):
        src, tgt = positive_edges[0, i].item(), positive_edges[1, i].item()
        positive_set.add((min(src, tgt), max(src, tgt)))  # Store as undirected
    
    negative_edges = []
    while len(negative_edges) < num_negative:
        src, tgt = np.random.choice(num_nodes, 2, replace=False)
        edge = (min(src, tgt), max(src, tgt))
        if edge not in positive_set and src != tgt:
            negative_edges.append([src, tgt])
    
    return torch.tensor(negative_edges).t()

def create_edge_features(edges, embeddings):
    """Create features for edge prediction by concatenating node embeddings"""
    edge_features = []
    for i in range(edges.shape[1]):
        node1, node2 = edges[0, i], edges[1, i]
        # Concatenate embeddings of the two nodes
        concat_emb = torch.cat([embeddings[node1], embeddings[node2]], dim=0)
        edge_features.append(concat_emb)
    return torch.stack(edge_features)

# Use a subset of the unique edges from our previous analysis
max_pos_edges = np.min([unique_edges.shape[1], 1000]) if USE_LAB_TRAINING else unique_edges.shape[1]
positive_edges = unique_edges[:, :max_pos_edges]
num_positive = positive_edges.shape[1]
print(f"Total positive edges: {num_positive}")

# Split positive edges into train/val/test
train_ratio, val_ratio, test_ratio = 0.7, 0.1, 0.2

train_size = int(train_ratio * num_positive)
val_size = int(val_ratio * num_positive)
test_size = num_positive - train_size - val_size

# Shuffle edges for random split
perm = torch.randperm(num_positive)
train_edges = positive_edges[:, perm[:train_size]]
val_edges = positive_edges[:, perm[train_size:train_size + val_size]]
test_edges = positive_edges[:, perm[train_size + val_size:]]

print(f"Positive edge splits:")
print(f"  Train: {train_edges.shape[1]}")
print(f"  Val: {val_edges.shape[1]}")
print(f"  Test: {test_edges.shape[1]}")

# Generate negative edges for each split
train_neg_edges = create_negative_edges(cora_data.num_nodes, train_edges, train_edges.shape[1])
val_neg_edges = create_negative_edges(cora_data.num_nodes, val_edges, val_edges.shape[1])
test_neg_edges = create_negative_edges(cora_data.num_nodes, test_edges, test_edges.shape[1])

print(f"Negative edge splits:")
print(f"  Train: {train_neg_edges.shape[1]}")
print(f"  Val: {val_neg_edges.shape[1]}")
print(f"  Test: {test_neg_edges.shape[1]}")

# Create features and labels for link prediction
print(f"\nCreating edge features...")

# Training data
X_train_pos = create_edge_features(train_edges, embeddings)
X_train_neg = create_edge_features(train_neg_edges, embeddings)
X_train_link = torch.cat([X_train_pos, X_train_neg], dim=0)
y_train_link = torch.cat([torch.ones(train_edges.shape[1]), torch.zeros(train_neg_edges.shape[1])], dim=0)

# Validation data
X_val_pos = create_edge_features(val_edges, embeddings)
X_val_neg = create_edge_features(val_neg_edges, embeddings)
X_val_link = torch.cat([X_val_pos, X_val_neg], dim=0)
y_val_link = torch.cat([torch.ones(val_edges.shape[1]), torch.zeros(val_neg_edges.shape[1])], dim=0)

# Test data
X_test_pos = create_edge_features(test_edges, embeddings)
X_test_neg = create_edge_features(test_neg_edges, embeddings)
X_test_link = torch.cat([X_test_pos, X_test_neg], dim=0)
y_test_link = torch.cat([torch.ones(test_edges.shape[1]), torch.zeros(test_neg_edges.shape[1])], dim=0)

print(f"\nLink prediction dataset shapes:")
print(f"  X_train_link: {X_train_link.shape} (features: concatenated embeddings)")
print(f"  y_train_link: {y_train_link.shape} (labels: 1=edge exists, 0=no edge)")
print(f"  X_val_link: {X_val_link.shape}")
print(f"  y_val_link: {y_val_link.shape}")
print(f"  X_test_link: {X_test_link.shape}")
print(f"  y_test_link: {y_test_link.shape}")

# Check class balance
train_positive_ratio = y_train_link.float().mean().item()
print(f"\nClass balance in training set:")
print(f"  Positive edges: {train_positive_ratio:.1%}")
print(f"  Negative edges: {1-train_positive_ratio:.1%}")

print(f"\nLink prediction data prepared!")

### 2.4.3 Link Prediction Neural Network

For link prediction, our neural network takes concatenated node embeddings as input and outputs a binary prediction (edge exists or not). The network architecture is similar to node classification but adapted for binary classification. We will use a single fully connect linear layer with a ReLU activation and the output layer. However, the output dimension is different and we use the sigmoid function as the last "layer" because, unlike the cross entropy loss function used in our node classification model, the binary cross entropy loss function used for this model does not handle logit inputs. 

<p style="font-size: 22px; color: Tomato"><strong>Excercise 2.3</strong></p>

In the code cell below, complete the implmentation of the `LinkPredictionClassifier` by:
- setting `self.fc2` to be a `tnn.Linear` layer with 1 output (for binary classifcation)
- completing the forward method by passing the input, `x` through the network layers

*Note, `tnn` is the import alias for `torch.nn`. The standard PyTorch convention is to use the alias `nn`. However, we will also import from `pytorch_geometric.nn` in future labs and want to avoid conflicts.

In [ ]:
# Define binary classifier for link prediction
class LinkPredictionClassifier(tnn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc1 = tnn.Linear(input_dim, hidden_dim)
        self.relu = tnn.ReLU()
        self.dropout = tnn.Dropout(0.2)
        self.fc2 = ___  # YOUR CODE HERE - Binary classification
        self.sigmoid = tnn.Sigmoid()  # Sigmoid for binary output
        
    def forward(self, x):
        # YOUR CODE HERE - Implement the forward pass
        x = ___
        x = ___
        x = tnn.Dropout(0.2)
        x = ___
        x = ___
        return x.squeeze()

# Model parameters
link_input_dim = X_train_link.shape[1]  # Concatenated embedding dimension (2 * embedding_dim)
link_hidden_dim = 64

print(f"=== Link Prediction Model ===")
print(f"Input dimension: {link_input_dim} (2 × {link_input_dim//2} concatenated embeddings)")
print(f"Hidden dimension: {link_hidden_dim}")
print(f"Output: Binary (edge exists or not)")

# Create model
link_model = LinkPredictionClassifier(link_input_dim, link_hidden_dim).to(device)
print(f"Model created with {sum(p.numel() for p in link_model.parameters()):,} parameters")
print(f"Model device: {next(link_model.parameters()).device}")

Below, we first create a method to handle model training. Some changes are necessary relative to the `train_classifier` model used to predict the node class membership. These differences occur because that was a multi-class prediction task, whereas the edge prediction task is a binary classification task. Hence, we require a different loss function. 

As with our previous models, we provide two sets of model and training parameters to support in class presentation (optimized for speed) and further exploration (optimized for performance). Toggle the `USE_LAB_TRAINING` boolean to switch between the two settings. You should expect better performance with the exploration parameters, especially if you have also used the exploration parameters to create higher quality node2vec embeddings.

In [ ]:
# Modified training function for binary classification
def train_binary_classifier(model, X_train, y_train, X_val, y_val, num_epochs, learning_rate=0.01, batch_size=64, verbose=True):
    """Train a binary classifier for link prediction"""
    
    # Get device from model
    device = next(model.parameters()).device
    
    # Ensure all data is on the correct device
    X_train = X_train.to(device)
    y_train = y_train.to(device).float()  # Binary labels need to be float for BCE loss
    X_val = X_val.to(device)
    y_val = y_val.to(device).float()
    
    criterion = tnn.BCELoss()  # Binary Cross Entropy for binary classification
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Create data loaders for batched training
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0
        num_batches = 0
        
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            train_out = model(batch_X)
            train_loss = criterion(train_out, batch_y)
            train_loss.backward()
            optimizer.step()
            
            epoch_train_loss += train_loss.item()
            num_batches += 1
        
        avg_train_loss = epoch_train_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation phase (only at end of epoch)
        model.eval()
        with torch.no_grad():
            val_out = model(X_val)
            val_loss = criterion(val_out, y_val)
            val_pred = (val_out > 0.5).float()  # Threshold at 0.5 for binary classification
            val_acc = (val_pred == y_val).float().mean()
        
        val_losses.append(val_loss.item())  
        val_accuracies.append(val_acc.item())
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d}: Train Loss = {avg_train_loss:.4f}, Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")
    
    return train_losses, val_losses, val_accuracies

# Train the link prediction model
print("=== Training Link Prediction Model ===")

link_num_epochs = 20 if USE_LAB_TRAINING else 30
link_learning_rate = 0.01
link_batch_size = 32 if USE_LAB_TRAINING else 64

print(f"Training parameters:")
print(f"  Epochs: {link_num_epochs}")
print(f"  Learning rate: {link_learning_rate}")
print(f"  Batch size: {link_batch_size}")

train_losses_link, val_losses_link, val_accuracies_link = train_binary_classifier(
    link_model, X_train_link, y_train_link, X_val_link, y_val_link, 
    link_num_epochs, link_learning_rate, link_batch_size,
    verbose=True
)

print(f"\nLink prediction training completed!")

Let's examine the model performance.

In [ ]:
# Evaluate link prediction model performance
print("=== Link Prediction Model Evaluation ===")

# Test predictions
link_model.eval()
with torch.no_grad():
    test_out_link = link_model(X_test_link.to(device))
    test_pred_link = (test_out_link > 0.5).float()
    test_acc_link = (test_pred_link == y_test_link.to(device)).float().mean()

print(f"Test accuracy: {test_acc_link:.4f}")

# Plot training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses_link, label='Training Loss', color='blue')
plt.plot(val_losses_link, label='Validation Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Link Prediction Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(val_accuracies_link, label='Validation Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Link Prediction Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# ROC curve for binary classification
from sklearn.metrics import roc_curve, auc
test_out_link_np = test_out_link.cpu().numpy()
y_test_link_np = y_test_link.numpy()

fpr, tpr, thresholds = roc_curve(y_test_link_np, test_out_link_np)
roc_auc = auc(fpr, tpr)

plt.subplot(1, 3, 3)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Link Prediction')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detailed classification report
test_pred_link_np = test_pred_link.cpu().numpy()
print("\nClassification Report:")
print(classification_report(y_test_link_np, test_pred_link_np, 
                          target_names=['No Edge', 'Edge'], digits=4))

# Confusion matrix for binary classification
cm = confusion_matrix(y_test_link_np, test_pred_link_np, normalize='pred')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=['No Edge', 'Edge'], 
            yticklabels=['No Edge', 'Edge'])
plt.title('Link Prediction Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()